# Full-corpus dense retrieval (Colab GPU)

Embeds all 511,958 EnterpriseRAG-Bench documents with a registered embedding model (BGE,
`gte-modernbert-base`, `arctic-embed-l-v2.0` or `qwen3-embedding-0.6b`; see
`src/config.py`) and scores the development split, like the BM25+ baseline.

1. **Runtime → Change runtime type → L4 GPU (or T4), and turn on High-RAM.** Standard RAM
   ran out during the titled bge-small run.
2. Run the cells in order. Setup takes about 10 minutes. Measured embedding times on a T4:
   1 h 50 min for bge-small, about 3–4 hours for bge-base. Estimates for
   `arctic-embed-l-v2.0`: about 5–7 hours on an L4 (bf16), 11–14 hours on a T4.
   The corpus is reused from Drive if it is already there.
3. Work is saved to Google Drive in shards, so after a disconnect rerun all cells and it
   resumes. Disconnect the runtime when finished so it stops using compute units.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Google Drive for resumable storage

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
WORK = "/content/drive/MyDrive/rag-dense"  # Corpus Parquet and embedding shards go here.
!mkdir -p {WORK}

## 2. Get the project code


In [ ]:
import os

BRANCH = "hira/dev"
if not os.path.exists("/content/project"):
    !git clone --depth 1 -b {BRANCH} https://github.com/hira-mian/Agentic-Enterprise-RAG-System.git /content/project
%cd /content/project
!git log -1 --format='%h %s'

In [ ]:
!pip install -q -r requirements.txt

## 3. Build the full corpus (skipped if already on Drive)

Clones the pinned upstream repository and converts it to Parquet (about 5–10 minutes).

In [ ]:
CORPUS = f"{WORK}/documents.parquet"
if not os.path.exists(CORPUS):
    !python -m src.data.corpus --clone --output {CORPUS}
else:
    print("Using existing", CORPUS)

## 4. Embed passages on the GPU

Each document is split into 500-token windows with 50-token overlap (about 1.7 million
passages). Completed shards are skipped when this cell is rerun.

- `MODEL` picks the embedding model. Its exact version and prefixes are recorded in the
  index `manifest.json` and reused automatically when searching.
- `TITLE_EVERY_PASSAGE = True` puts each document's title at the start of every passage
  (not just the first), as in the BGE-base run and the model screening.
- Precision is bf16 on GPUs that support it (L4, A100) and fp16 otherwise (T4). If the GPU
  runs out of memory, the batch size is halved automatically.

Each combination writes to its own Drive folder, so earlier indexes are kept.

In [ ]:
import json
import torch

MODEL = "bge-base"  # Team default; others: "arctic-embed-l-v2.0", "qwen3-embedding-0.6b"
TITLE_EVERY_PASSAGE = True

NAME = ("dense" if MODEL == "bge-small" else f"dense_{MODEL.removeprefix('bge-')}") + (
    "_titled" if TITLE_EVERY_PASSAGE else "")
FLAG = "--title-every-passage" if TITLE_EVERY_PASSAGE else ""
RESULTS = f"{NAME}_full_development"
NEW_GPU = torch.cuda.get_device_capability()[0] >= 8  # Ampere or newer, e.g. L4 or A100.
PRECISION = "bf16" if NEW_GPU else "fp16"
SMALL = MODEL in ("bge-small", "bge-base", "gte-modernbert-base")
BATCH = (256 if NEW_GPU else 128) if SMALL else (64 if NEW_GPU else 32)
# Smaller shards use less RAM. A started index keeps its shard size and precision, so it
# can resume on a different GPU type without mixing settings.
existing = f"{WORK}/{NAME}/manifest.json"
SHARD_DOCS = 5000
if os.path.exists(existing):
    started = json.load(open(existing))
    SHARD_DOCS = started["shard_docs"]
    PRECISION = "bf16" if started["model"].get("bf16") else ("fp16" if started["model"]["fp16"] else "fp32")
print(torch.cuda.get_device_name(), "| precision", PRECISION, "| batch", BATCH, "| shard docs", SHARD_DOCS)
print("Index folder:", f"{WORK}/{NAME}", "| results:", RESULTS)

!python -m src.retrieval.corpus_dense --corpus {CORPUS} --output {WORK}/{NAME} --model {MODEL} \
    --device cuda --batch-size {BATCH} --precision {PRECISION} --shard-docs {SHARD_DOCS} {FLAG}

## 5. Score the development split

In [ ]:
!python -m src.evaluation.run_retrieval --method dense --index {WORK}/{NAME} --device cuda --split development \
    --questions {WORK}/questions.jsonl --output evaluation/results/{RESULTS}

In [ ]:
import json

def load(name):
    return json.load(open(f"evaluation/results/{name}/summary.json"))

# Compare bge-small with its original run, and other models with titled BGE-base.
if MODEL == "bge-small":
    BEFORE = "dense_full_development"
elif MODEL == "bge-base":
    BEFORE = "dense_titled_full_development"
else:
    BEFORE = "dense_base_titled_full_development"
new, old, bm25 = load(RESULTS), load(BEFORE), load("bm25_full_development")
print(f"before = {BEFORE}, after = {RESULTS}")
print(f"{'type':26} {'scored':>6} {'before':>8} {'after':>8} {'change':>8} {'BM25+':>7}   (Recall@10)")
rows = [("Overall", "overall")] + [(t, t) for t in new["question_types"]]
for label, key in rows:
    pick = (lambda s: s["overall"]) if key == "overall" else (lambda s: s["question_types"][key])
    a, b, c = pick(old)["recall@10"]["mean"], pick(new)["recall@10"]["mean"], pick(bm25)["recall@10"]["mean"]
    if b is None:
        continue
    print(f"{label:26} {pick(new)['recall@10']['scored']:>6} {a:>8.3f} {b:>8.3f} {b - a:>+8.3f} {c:>7.3f}")
print()
for metric in ("recall@5", "recall@20", "ndcg@10"):
    a, b = old["overall"][metric]["mean"], new["overall"][metric]["mean"]
    print(f"{metric:>10}: before {a:.3f} -> after {b:.3f} ({b - a:+.3f})")

## 6. Save the results

This copies the result files to Drive (`rag-dense/results`) and downloads a zip.
Commit the results folder to `evaluation/results/`, or share the Drive folder.

In [ ]:
!mkdir -p {WORK}/results && cp -r evaluation/results/{RESULTS} {WORK}/results/
!cd evaluation/results && zip -qr /content/{RESULTS}.zip {RESULTS}
from google.colab import files
files.download(f"/content/{RESULTS}.zip")